In [2]:
import faiss
from faker import Faker
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
fake = Faker()
Faker.seed(42)
corpus = [fake.paragraph(nb_sentences=3) for _ in range(1000)]
print(corpus[723])

Well commercial nice next. Time live relationship as. Assume rest now water road require.


Custom cossim approach

In [7]:
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(corpus)
feature_names = vectorizer.get_feature_names_out()

# This returns a sparse matrix of TF-IDF vectors
df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)

In [10]:
def get_most_similar_sentence(user_query: str, tfidf_df: pd.DataFrame, vectorizer: TfidfVectorizer, corpus: list[str]) -> str:
    """
    Returns the sentence that is the most similar to the user query (by tf-idf cossine similarity)
    """
    user_query_tf_idf = vectorizer.transform(user_query)
    cossine_similarities = np.dot(user_query_tf_idf.toarray(), df.values.T)
    max_similarity_idx = np.argmax(cossine_similarities)
    print(corpus[max_similarity_idx])
    print(f"Cossine Similarity: {cossine_similarities[0, max_similarity_idx]:.2f}")

In [11]:
sentence = "leave the computer in the kitchen."
get_most_similar_sentence(user_query=[sentence], tfidf_df=df, vectorizer=vectorizer, corpus=corpus)

Debate leave each. Kitchen drop computer left political sound study the.
Cossine Similarity: 0.56


FAISS Vector Database approach

In [4]:
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(corpus).toarray().astype('float32')


d = tfidf_matrix.shape[1] 
index = faiss.IndexFlatL2(d)  # FlatL2 since our database is small (so we don't need to cluster)
index.add(tfidf_matrix)


def search_vector_db(query, vectorizer=vectorizer, knn=3):
    query_vector = vectorizer.transform([query]).toarray().astype('float32')
    distances, indices = index.search(query_vector, knn)
    
    print(f"Query: '{query}'\n" + "-"*30)
    for i, idx in enumerate(indices[0]):
        print(f"Match {i+1} (Dist: {distances[0][i]:.2f}):")
        print(f"Text: {corpus[idx]}\n")

In [6]:
search_vector_db("leave the computer in the kitchen.")

Query: 'leave the computer in the kitchen.'
------------------------------
Match 1 (Dist: 0.87):
Text: Debate leave each. Kitchen drop computer left political sound study the.

Match 2 (Dist: 1.31):
Text: Despite whole computer will Mr without.

Match 3 (Dist: 1.46):
Text: Fact huge fight still leave.

